# Train the design critic yourself

**What you are about to build.** Stratum makes 648 designs for every post. Nobody
wants to look at 648. The *critic* is a small machine-learning model that scores each
design, so the app can show the best three.

**How a model can judge a design.** It cannot look at the picture the way you do.
Instead, each design is described with **35 numbers** (we call them *features* or
*measurements*): how well things line up, how close words get to the edge, how
readable the text is, how big the headline is next to the offer, and so on. The model
learns which combinations of those numbers mean "good".

**How it learns.** We take good designs and **break copies of them on purpose**:
push a line out of alignment, shrink the small print, drop the dark band behind the
words. Then we show the model thousands of these groups and say: *the unbroken one
should get the highest score.* Learning to put things in the right order is called
**learning to rank**.

**The steps**

| step | what you do | time |
|---|---|---|
| 0 | set up | 1 min |
| 1 | look at one design, its 35 numbers, and a broken copy | 5 min |
| 2 | make the training data | 2 min |
| 3 | build the simple baseline to beat | 5 min |
| 4 | train the model | 3 min |
| 5 | choose how big the model should be | 5 min |
| 6 | the final exam, on a brand the model has never seen | 3 min |
| 7 | see what it learned, and where it fails | 5 min |
| 8 | save the model the app will use | 2 min |
| 9 | write down your results | 2 min |
| 10 | practise explaining it | 10 min |

**How to use this notebook.** Click into a grey code cell and press **Shift + Enter**
to run it. Go top to bottom, one cell at a time. Read the text above each cell first:
it says what the code does and what you should see. Your numbers may differ a little
from mine in the last decimal places (different computer, different processor); that
is fine.

## Step 0 — Set up

This cell imports the libraries and the parts of Stratum we need:

- `numpy` does fast maths on tables of numbers;
- `pandas` shows those tables nicely;
- `lightgbm` is the machine-learning library (made by Microsoft, open source, MIT licence);
- `stratum_worker.critic` is the "lab equipment" already in the repo: it measures
  designs, breaks them on purpose, and builds the training data.

In [ ]:
import inspect
import json
import random
import tempfile
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stratum_spec import retarget
from stratum_worker.critic import damage
from stratum_worker.critic.data import SPLITS, groups, kits
from stratum_worker.critic.features import FEATURES, measure
from stratum_worker.critic.ranker import DEFAULT_MODEL, Forest, heuristic, save_model
from stratum_worker.design import lay_out
from stratum_worker.design.preview import SAMPLE_COPY, bind, ink, render, stand_ins

# The notebook runs from ml/critic/, so the repository root is two folders up.
ROOT = Path.cwd().resolve()
while not (ROOT / "templates").is_dir():
    ROOT = ROOT.parent
print("repository:", ROOT)
print("lightgbm version:", lgb.__version__)
print("number of measurements per design:", len(FEATURES))

**You should see** the path of your `stratum` folder, a LightGBM version starting
with `4.`, and `35`.

If you get `ModuleNotFoundError`, the notebook was not started with `make notebook`
from the `stratum` folder. Close it and start it that way.

## Step 1 — Look at one design, its 35 numbers, and a broken copy

First we make one design for a fictional plumber, Kowalski Plumbing, and draw it.

- `lay_out` lays a design out around the words (Two photos, panel, and so on). The
  id `ds.panel.1.0000000` means: family *panel*, first choice of every option.
- `bind` puts the words and stand-in pictures into it.
- `render` draws it as a small picture.

The photos are made up by the computer (coloured blobs), so no real business's photo
is ever used.

In [ ]:
KIT = kits(ROOT)["kowalski-plumbing"]
LOGO = ROOT / "clients" / "kowalski-plumbing" / "logo.png"
FILES = stand_ins(Path(tempfile.mkdtemp()), LOGO)  # stand-in photos + the logo

laid = lay_out("ds.panel.1.0000000", KIT, ROOT, copy=SAMPLE_COPY)
design = bind(laid, KIT, SAMPLE_COPY, logo=ink(LOGO))
good = retarget(design, "4:5")  # the feed-post size
render(good, KIT, ROOT, FILES, width=270)

**You should see** a poster: a blob "photo" on top with the KOWALSKI logo, a navy
panel with an orange accent line, the headline, the offer line and "Book your place
today".

Now the 35 numbers for that poster. A few to read:

- `margin_min` about **0.074**: the closest any text gets to an edge is 7.4% of the
  poster's short side. That is the margin the designs are built with.
- `contrast_min` about **14**: the least readable text is at 14:1. Readable text
  needs about 4.5:1 (a web accessibility rule called WCAG).
- `align_misfit` **0**: every element lines up with something else.
- `headline_size` about **0.06**: the headline letters are 6% of the short side.
  (The panel is small in this design, so the headline had to shrink to fit.)

In [ ]:
numbers = pd.Series(measure(good, KIT, ROOT), index=FEATURES).round(3)
numbers.to_frame("value")

Now **break it on purpose**. `damage.misalign` nudges one line of text sideways.
The `3` is the strength (1 = slightly, 2 = clearly, 3 = badly). Then we measure again
and show only the numbers that changed.

In [ ]:
broken = damage.misalign(good, 3, random.Random(1), KIT)
changed = pd.DataFrame(
    {
        "good": measure(good, KIT, ROOT),
        "broken": measure(broken, KIT, ROOT),
    },
    index=FEATURES,
).round(3)
display(render(broken, KIT, ROOT, FILES, width=270))
changed[changed.good != changed.broken]

**You should see** the headline pushed left, right up against the edge, and a small
table in which `align_misfit` and `left_edge_spread` went up from 0 and `margin_min`
dropped from 0.074 to about 0 (the words now touch the edge).
That is the whole idea: **a mistake a designer would see shows up as numbers
changing.**

*Try this (optional):* change `damage.misalign` to `damage.tiny_type`,
`damage.low_contrast` or `damage.collide` and run the cell again. Watch which numbers
move each time.

## Step 2 — Make the training data

**A group** is one design plus broken copies of it. There are two kinds:

| kind | what is in it | labels (higher = better) | the question it teaches |
|---|---|---|---|
| broken | the good design + one copy for each kind of damage | good = 1, every broken copy = 0 | is any damage worse than none? |
| severity | the good design + one kind of damage at 3 strengths | 3, 2, 1, 0 | is more of the same damage worse? |

In "broken" groups we never say that a crooked line is worse than small print,
because nobody could say that in general.

**The three splits** are like school:

- **train** is the textbook the model learns from: three brands, copy 0 to 9;
- **valid** is the practice exam we use to make choices: a fourth brand and new copy;
- **test** is the final exam, used **once** at the end: a fifth brand and new copy.

The model never sees the exam brands or their words while learning. If it scores
well there, it learned design rules, not the answers by heart. Splitting by brand and
by words, not by random rows, is what makes the result honest.

This cell builds all of it. **It takes about a minute** because it lays out,
measures and breaks thousands of posters.

In [ ]:
started = time.time()
train_split, valid_split, test_split = SPLITS
train = list(groups(ROOT, train_split, designs_per_copy=40, seed=0))
valid = list(groups(ROOT, valid_split, designs_per_copy=40, seed=0))
test = list(groups(ROOT, test_split, designs_per_copy=40, seed=0))
print(f"train {len(train)} groups, valid {len(valid)}, test {len(test)}")
print(f"({time.time() - started:.0f} s)")
print("brands in train:", train_split.kits)
print("brand in valid:", valid_split.kits, "  brand in test:", test_split.kits)

**You should see** `train 2940 groups, valid 294, test 294`.

Here is one "broken" group from the training data: the first row is the good design,
every other row is a broken copy, and the `label` column is what the model should
learn to reproduce as an order.

In [ ]:
one = next(g for g in train if g.kind == "broken")
table = pd.DataFrame(one.features, columns=FEATURES)
table.insert(0, "damage", [d or "(untouched)" for d in one.damages])
table.insert(1, "label", one.labels)
table[
    ["damage", "label", "align_misfit", "margin_min", "contrast_min", "offer_size", "text_overlap"]
].round(3)

## Step 3 — The baseline: a formula written by hand

Before training anything, build the simplest thing that could work. This is a
**baseline**. Any model has to beat it, or it is not worth having.

Ours is a formula with penalties someone chose by judgement: minus points for low
contrast, for text near the edge, for overlapping words, and so on. Read it:

In [ ]:
print(inspect.getsource(heuristic))

Now **three ways to mark an answer**. All of them work group by group:

1. **base_wins**: in a "broken" group, for each broken copy, did the good design get
   a higher score? (Share of yes answers. 1.0 is perfect; 0.5 is a coin toss.)
2. **top1**: in a "broken" group, did the good design come **first** of all of them?
3. **order**: in a "severity" group, take every pair; is the less-broken one scored
   higher? (Share of pairs in the right order.)

In [ ]:
def pairs_in_order(scores, labels):
    right = total = 0.0
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            if labels[i] == labels[j]:
                continue
            total += 1
            better, worse = (i, j) if labels[i] > labels[j] else (j, i)
            right += (
                1.0
                if scores[better] > scores[worse]
                else 0.5
                if scores[better] == scores[worse]
                else 0.0
            )
    return right, total


def marks(score_fn, data):
    wins = n = firsts = n_broken = order_right = order_total = 0.0
    for g in data:
        s = score_fn(g.features)
        if g.kind == "broken":
            wins += (s[0] > s[1:]).sum() + 0.5 * (s[0] == s[1:]).sum()
            n += len(s) - 1
            firsts += float(np.argmax(s) == 0)
            n_broken += 1
        else:
            r, t = pairs_in_order(s, g.labels)
            order_right += r
            order_total += t
    return {"base_wins": wins / n, "top1": firsts / n_broken, "order": order_right / order_total}


pd.Series(marks(heuristic, valid)).round(3).to_frame("hand-written rule (valid)")

**You should see** about: base_wins **0.87**, top1 **0.29**, order **0.89**.

Why is `top1` so low when base_wins looks decent? The formula has no penalty for some
kinds of damage (for example a call-to-action that drifted a little), and it treats
tiny differences as real. In a group of ten copies, one of them usually edges past
the good design. That is exactly the kind of thing a trained model can fix.

## Step 4 — Train the model

**What LightGBM builds: many small decision trees.**

A *decision tree* is a flowchart of yes/no questions about the numbers, such as "is
`contrast_min` below 4.2? is `margin_min` below 0.05?". At the end of each path is a
score. One small tree is a weak judge.

*Boosting* builds trees one after another. Each new tree is trained to fix the
mistakes the trees before it still make. The final score is the sum of all the trees.

*LambdaRank* is how the mistakes are counted: not "is this score right?" but "is the
**order inside each group** right?". That is why we tell it the group sizes.

The settings, in plain words:

- `learning_rate` 0.05: each tree only nudges the score a little, which is slower
  but steadier;
- `num_leaves` 15: each tree has at most 15 end points, so it stays small;
- `bagging` and `feature_fraction` 0.8: each tree sees a random 80% of the rows and
  measurements, so no tree can lean on one quirk;
- `seed` 0 and `deterministic`: the same result every time you run it.

We train 400 trees; step 5 decides how many to keep. It only takes a few seconds.

In [ ]:
def stack(data):
    x = np.vstack([g.features for g in data])
    y = np.concatenate([g.labels for g in data])
    return x, y, [len(g.labels) for g in data]


x_train, y_train, sizes_train = stack(train)
params = {
    "objective": "lambdarank",
    "learning_rate": 0.05,
    "num_leaves": 15,
    "min_data_in_leaf": 20,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "feature_fraction": 0.8,
    "lambda_l2": 1.0,
    "seed": 0,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 1,
    "verbosity": -1,
}
started = time.time()
booster = lgb.train(
    params,
    lgb.Dataset(x_train, y_train, group=sizes_train, feature_name=list(FEATURES)),
    num_boost_round=400,
)
print(
    f"trained {booster.num_trees()} trees on {len(x_train)} rows in {time.time() - started:.0f} s"
)

model_marks = marks(lambda x: booster.predict(x), valid)
pd.DataFrame(
    {"hand-written rule": marks(heuristic, valid), "your model (400 trees)": model_marks}
).round(3)

**You should see** your model well ahead of the rule on the practice exam
(valid): base_wins about **0.99**, top1 about **0.88** and order about **0.95**,
against the rule's 0.87, 0.29 and 0.89.

You just trained a machine-learning model.

## Step 5 — How many trees? (not too few, not too many)

- **Too few trees** and the model is too simple: it misses patterns (*underfitting*).
- **Too many** and it starts memorising quirks of the training brands that do not
  carry over to new ones (*overfitting*), and it gets bigger and slower.

We let the **practice exam (valid)** decide. We score the model using only the first
25, 50, 100, 200 and 400 trees, and keep the **smallest** model that is within 0.002
of the best. Smaller is faster and less likely to be memorising.

In [ ]:
rows = {}
for k in (25, 50, 100, 200, 400):
    m = marks(lambda x, k=k: booster.predict(x, num_iteration=k), valid)
    rows[k] = {**m, "average": (m["base_wins"] + m["order"]) / 2}
curve = pd.DataFrame(rows).T.round(4)
best_avg = curve["average"].max()
TREES = int(min(k for k in curve.index if curve.loc[k, "average"] >= best_avg - 0.002))
print("keeping", TREES, "trees")
curve.plot(
    y="average",
    marker="o",
    figsize=(5, 3),
    title="practice-exam score vs number of trees",
    legend=False,
)
plt.xlabel("trees")
plt.ylabel("average of base_wins and order")
plt.show()
curve

**You should see** the score climb quickly and then flatten: about 0.94 at 25 trees,
about 0.966 at 200, and barely more at 400. So you keep **200 trees**.

## Step 6 — The final exam (run this once)

Now the **test** split: a brand (Sparkle Squad) and words the model has never seen,
and that we did not use to make any choice. We run it **once**. If you kept changing
things until the test score looked good, the test would stop being a fair exam.

**Confidence intervals.** The test has 294 groups, and a different 294 would give
slightly different numbers. So we *bootstrap*: we pick 294 groups at random from the
test set, allowing repeats, score them, and do that 1,000 times. The middle 95% of
those scores is the range in which the true number very likely lies. If the model's
range and the rule's range do not overlap, the win is real, not luck.

In [ ]:
def with_range(score_fn, data, repeats=1000, seed=0):
    rng = np.random.default_rng(seed)
    point = marks(score_fn, data)
    draws = [
        marks(score_fn, [data[i] for i in rng.integers(0, len(data), len(data))])
        for _ in range(repeats)
    ]
    return {
        name: (
            round(point[name], 3),
            round(np.percentile([d[name] for d in draws], 2.5), 3),
            round(np.percentile([d[name] for d in draws], 97.5), 3),
        )
        for name in point
    }


def critic(x):
    return booster.predict(x, num_iteration=TREES)


started = time.time()
exam = {"your model": with_range(critic, test), "hand-written rule": with_range(heuristic, test)}
print(f"({time.time() - started:.0f} s)  each cell: (score, low end, high end) of the 95% range")
pd.DataFrame(exam)

**It takes about 45 seconds** (1,000 re-scorings of two models). **You should see**,
close to:

| | your model | hand-written rule |
|---|---|---|
| base_wins | 0.993 (0.989 to 0.997) | 0.847 (0.829 to 0.866) |
| top1 | 0.932 (0.893 to 0.972) | 0.245 (0.178 to 0.316) |
| order | 0.946 (0.930 to 0.960) | 0.858 (0.826 to 0.889) |

The ranges do not overlap, so the model really is better than the rule, on a brand
it has never seen.

## Step 7 — What did it learn, and where does it fail?

**Which measurements it relied on.** *Gain* adds up how much each measurement helped
the trees make better splits. Bigger means it mattered more.

In [ ]:
gain = pd.Series(booster.feature_importance("gain", iteration=TREES), index=FEATURES)
gain = (gain / gain.sum()).sort_values()
gain.tail(12).plot.barh(figsize=(5, 4), title="what the model relied on (share of gain)")
plt.show()

**You should see** readability (`contrast_min`), alignment (`align_misfit`), spacing
(`gap_unevenness`, `gap_min`, `margin_min`) and the logo overlapping text near the
top. Those are the things a designer checks first, and the model found them from
examples alone.

**Where it fails.** For each kind of damage: how often did the good design beat the
broken copy on the final exam?

In [ ]:
def wins_by_damage(score_fn, data):
    out = {}
    for g in data:
        if g.kind != "broken":
            continue
        s = score_fn(g.features)
        for row, name in enumerate(g.damages):
            if name:
                out.setdefault(name.split(":")[0], []).append(float(s[0] > s[row]))
    return pd.Series({k: np.mean(v) for k, v in out.items()})


pd.DataFrame(
    {"your model": wins_by_damage(critic, test), "rule": wins_by_damage(heuristic, test)}
).round(3)

**You should see** your model near 1.0 on almost everything. Its weakest kinds are
usually `unbalance` (the whole text block slid up or down) and `misalign` at the
lightest strength, which are the subtlest to spot. The rule's worst is
`strip_backing` (the dark band behind the words removed): it scores under 0.5, which
is worse than a coin toss, because it has no idea how busy the photo behind the words
is.

**Say this out loud**, because it matters: this model is a **defect detector**. It
learned "broken versus fine". It has **not** learned *taste*, which of two good
designs a person would prefer. That needs real people's choices, which is the next
part of the plan.

## Step 8 — Save the model the app will use

The server does not run LightGBM. We save the trees as a JSON file, and Stratum
walks through them with plain `numpy`. It is faster to start, and there is one less
library to install on the server.

`save_model` does three things:

1. it writes only what prediction needs (about 540 KB);
2. it stores a few rows and LightGBM's own scores for them inside the file, as proof;
3. it **refuses to save** if the numpy version disagrees with LightGBM. A test in the
   repository re-checks that proof every time `make check` runs.

In [ ]:
results = {name: {k: list(v) for k, v in scores.items()} for name, scores in exam.items()}
proof_x, _, _ = stack(test[:12])
forest = save_model(
    booster,
    TREES,
    DEFAULT_MODEL,
    proof_x=proof_x,
    meta={
        "model": "critic-v1",
        "trained_by": "Rushil",
        "trained_on": time.strftime("%Y-%m-%d"),
        "trees_chosen_on_valid": TREES,
        "test": results,
    },
)
print("saved:", DEFAULT_MODEL.relative_to(ROOT), f"({DEFAULT_MODEL.stat().st_size / 1024:.0f} KB)")

# The app's own loader, scoring the good design from step 1 against its broken copy:
app_model = Forest.load(DEFAULT_MODEL)
print("good design scores  ", round(float(app_model.predict(measure(good, KIT, ROOT)[None])[0]), 3))
print(
    "broken copy scores  ", round(float(app_model.predict(measure(broken, KIT, ROOT)[None])[0]), 3)
)

**You should see** the file saved at
`apps/worker/src/stratum_worker/critic/model/critic-v1.json`, about 540 KB, and the
good design scoring clearly higher than the broken copy (about 3.7 against -4.6).
Scores are not percentages; only which one is higher matters.

## Step 9 — Write down your results

This cell writes your numbers into `ml/critic/report-v1.json` and fills in the model
card, `ml/critic/MODEL_CARD.md`, with **your** results. Open the card afterwards and
read it: it is the page someone reviewing your GitHub will read about this model.

In [ ]:
report = {
    "model": "critic-v1",
    "trained_by": "Rushil",
    "trees": TREES,
    "data": {"train": len(train), "valid": len(valid), "test": len(test)},
    "validation_curve": curve["average"].to_dict(),
    "test": results,
    "test_by_damage": {
        "model": wins_by_damage(critic, test).round(4).to_dict(),
        "rule": wins_by_damage(heuristic, test).round(4).to_dict(),
    },
    "importance": gain.sort_values(ascending=False).round(4).to_dict(),
}
report_path = ROOT / "ml" / "critic" / "report-v1.json"
report_path.write_text(json.dumps(report, indent=2) + "\n")

card = (ROOT / "ml" / "critic" / "MODEL_CARD.template.md").read_text()
m, r = exam["your model"], exam["hand-written rule"]
fill = {
    "TREES": TREES,
    "DATE": time.strftime("%d %b %Y"),
    "M_WINS": m["base_wins"],
    "R_WINS": r["base_wins"],
    "M_TOP1": m["top1"],
    "R_TOP1": r["top1"],
    "M_ORDER": m["order"],
    "R_ORDER": r["order"],
    "TOP_FEATURES": ", ".join(
        f"{k} {v:.2f}" for k, v in gain.sort_values(ascending=False).head(6).items()
    ),
}
for key, value in fill.items():
    text = (
        value
        if not isinstance(value, tuple)
        else f"{value[0]:.3f} [{value[1]:.3f}, {value[2]:.3f}]"
    )
    card = card.replace("{" + key + "}", str(text))
(ROOT / "ml" / "critic" / "MODEL_CARD.md").write_text(card)
print("wrote ml/critic/report-v1.json and ml/critic/MODEL_CARD.md")

## Step 10 — Explain it (practise these out loud)

**What does the critic do?**
It scores poster designs so the app can show the best few out of hundreds. It uses
35 measurements a designer would check: alignment, margins, spacing, contrast,
headline-to-offer size, and whether words sit on a busy photo.

**Why learning to rank, and not "good or bad"?**
The product never needs an absolute verdict. It needs to pick the best three out of
many, so what matters is order within one post. LambdaRank optimises exactly that.

**Where did the training data come from?**
I generated it. Good designs come from our design space; then I broke copies of them
on purpose in ten ways, at three strengths. All of it is fictional brands, fictional
words and computer-made photos, so there are no data-licence problems.

**How do you know it is not memorising?**
I split by brand and by copy, never by row. The test brand and its words were never
seen in training or used for any choice, and I ran the test once.

**What is the baseline, and did you beat it?**
A hand-written penalty formula. On the unseen brand the model ranks the untouched
design first 93% of the time; the formula manages 25%. The 95% bootstrap ranges do
not overlap.

**What are its limits?**
It is a defect detector, not taste: it learned "broken versus fine". The photos are
synthetic, and it scores one size at a time.

**Why is there no ML library on the server?**
The model is JSON trees evaluated with numpy. The file carries LightGBM's own scores
for a few rows, and a test checks that numpy reproduces them exactly.

**What is next?**
Taste. First, an AI vision model compares pairs of good designs, asked twice with
the positions swapped. Then people compare pairs on a labelling page, and designers'
real picks in the app are logged. The held-out human pairs become the real test.

## Step 11 — Save it to GitHub

Close this notebook first (File → Shut Down), then run the block in Warp from the
`stratum` folder that I gave you in the chat. It commits your model, your report,
your model card and this notebook with your outputs in it, so anyone reading the
repository can see that you ran it and what you got.